In [1]:
# ==========================================================
# ACTIVIDAD 20. Métricas formales de generalización
# (a) ¿Un K-Means entrenado desde cero en cada celda encuentra
#     los mismos regímenes? (ARI y distancia entre centros)
# (b) ¿Las empresas reservadas entran y salen del Estrés en las
#     mismas fechas que las de entrenamiento?
# (c) Evolución trimestral en todo el periodo, prueba incluida.
# El modelo definitivo NO se modifica.
# ==========================================================
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from google.colab import drive
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

SEMILLA_REAJUSTE = 7               # distinta de 42: así la celda de entrenamiento sirve de referencia
COLUMNAS_Z = ["z_volatilidad_21", "z_cambio_volatilidad", "z_rendimiento_21",
              "z_caida_maximo_63", "z_volumen_anormal"]

drive.mount("/content/drive")
RAIZ = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# ----------------------------------------------------------
# 1. Modelo definitivo y datos
# ----------------------------------------------------------
guardado = joblib.load(RAIZ / "02_modelos" / "kmeans_k3.joblib")
modelo, renumerar, nombres = guardado["modelo"], guardado["renumerar"], guardado["nombres"]
k = modelo.n_clusters

df = pd.read_parquet(RAIZ / "01_datos" / "variables_preparadas.parquet",
                     columns=["date", "grupo", "bloque"] + COLUMNAS_Z)
for col in ["grupo", "bloque"]:
    df[col] = df[col].astype(str)
df["regimen"] = pd.Series(modelo.predict(df[COLUMNAS_Z].to_numpy()), index=df.index).map(renumerar)
df["celda"] = df["grupo"] + " / " + df["bloque"]
ORDEN_CELDAS = ["entrenamiento / entrenamiento", "entrenamiento / validacion", "entrenamiento / prueba",
                "reservada / entrenamiento", "reservada / validacion", "reservada / prueba"]

# Centros del modelo definitivo en el orden Calma, Impulso, Estrés
orden = np.argsort(modelo.cluster_centers_[:, 0])
centros_modelo = modelo.cluster_centers_[orden]

# ----------------------------------------------------------
# (a) Reentrenar desde cero en cada celda y comparar
# ----------------------------------------------------------
filas = []
for celda in ORDEN_CELDAS:
    datos = df[df["celda"] == celda]
    X = datos[COLUMNAS_Z].to_numpy()
    nuevo = KMeans(n_clusters=k, n_init=10, random_state=SEMILLA_REAJUSTE).fit(X)
    centros_nuevos = nuevo.cluster_centers_[np.argsort(nuevo.cluster_centers_[:, 0])]
    distancias = np.linalg.norm(centros_nuevos - centros_modelo, axis=1)   # en desviaciones típicas
    filas.append({"celda": celda, "filas": len(X),
                  "ari": adjusted_rand_score(nuevo.labels_, datos["regimen"]),
                  **{f"distancia_{nombres[i + 1]}": d for i, d in enumerate(distancias)}})
    print(f"{celda} listo")

estabilidad = pd.DataFrame(filas).set_index("celda")
print("\n=== (a) MODELO REENTRENADO EN CADA CELDA FRENTE AL MODELO DEFINITIVO ===")
print("ARI: 1 = agrupa igual | distancia: entre centros equivalentes, en desviaciones típicas")
print(estabilidad)

# ----------------------------------------------------------
# (b) Sincronía diaria entre empresas de entrenamiento y reservadas
# ----------------------------------------------------------
diario = (pd.crosstab([df["date"], df["grupo"]], df["regimen"], normalize="index") * 100).unstack("grupo")
bloque_fecha = df.groupby("date")["bloque"].first()
filas = []
for regimen, nombre in nombres.items():
    a = diario[(regimen, "entrenamiento")]
    b = diario[(regimen, "reservada")]
    for bloque in ["entrenamiento", "validacion", "prueba"]:
        fechas = bloque_fecha.index[bloque_fecha == bloque]
        filas.append({"regimen": nombre, "bloque": bloque,
                      "correlacion": a[fechas].corr(b[fechas]),
                      "dif_media_pp": (a[fechas] - b[fechas]).abs().mean()})
sincronia = pd.DataFrame(filas)
print("\n=== (b) % DIARIO DE ACCIONES EN CADA RÉGIMEN: ENTRENAMIENTO FRENTE A RESERVADAS ===")
print("correlación entre las dos series diarias | dif_media_pp: diferencia media en puntos porcentuales")
print(sincronia.pivot(index="regimen", columns="bloque", values=["correlacion", "dif_media_pp"])
      .reindex(list(nombres.values())))

# ----------------------------------------------------------
# (c) Evolución trimestral de Impulso y Estrés en todo el periodo
# ----------------------------------------------------------
trimestral = (pd.crosstab([df["date"].dt.to_period("Q"), df["grupo"]], df["regimen"], normalize="index")
              * 100).unstack("grupo")[[2, 3]]
trimestral.columns = [f"{nombres[r]}_{g[:3]}" for r, g in trimestral.columns]
print("\n=== (c) % DE FILAS EN IMPULSO Y ESTRÉS POR TRIMESTRE (ent = entrenamiento, res = reservadas) ===")
print(trimestral)

# ----------------------------------------------------------
# Guardar
# ----------------------------------------------------------
estabilidad.to_csv(RAIZ / "02_modelos" / "generalizacion_estabilidad.csv")
sincronia.to_csv(RAIZ / "02_modelos" / "generalizacion_sincronia.csv", index=False)
trimestral.to_csv(RAIZ / "02_modelos" / "regimenes_trimestrales.csv")
print("\nGuardado: generalizacion_estabilidad.csv, generalizacion_sincronia.csv y regimenes_trimestrales.csv")

Mounted at /content/drive
entrenamiento / entrenamiento listo
entrenamiento / validacion listo
entrenamiento / prueba listo
reservada / entrenamiento listo
reservada / validacion listo
reservada / prueba listo

=== (a) MODELO REENTRENADO EN CADA CELDA FRENTE AL MODELO DEFINITIVO ===
ARI: 1 = agrupa igual | distancia: entre centros equivalentes, en desviaciones típicas
                                filas   ari  distancia_Calma  distancia_Impulso  distancia_Estrés
celda                                                                                            
entrenamiento / entrenamiento  815829 0.978            0.015              0.024             0.004
entrenamiento / validacion     488607 0.754            0.347              0.313             1.177
entrenamiento / prueba         392889 0.663            0.327              0.272             0.913
reservada / entrenamiento      350374 0.939            0.037              0.059             0.123
reservada / validacion         209842 0.7